In [3]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

NAMES = [name.strip() for name in open("names.txt", "r").readlines()]

def name_to_tensor(name):
    return torch.tensor([0]+[1 + ord(c) - ord("a") for c in name]+[0])
def tensor_to_name(tensor):
    return "".join([chr(c + ord("a")-1) for c in tensor[1:-1]])

In [4]:
inputX = []
inputY = []

CONTEXT_WIDTH = 3
#DEVICE = "cpu"
DEVICE = "cuda"

for name in NAMES:
    t = name_to_tensor(name)
    x = [0] * CONTEXT_WIDTH

    for i in range(0, len(t)-1):
        x = x[1:] + [t[i].item()]
        inputX.append(x)
        inputY.append(t[i+1].item())

inputX = torch.tensor(inputX, device=DEVICE)
inputY = torch.tensor(inputY, device=DEVICE)

#print(inputX)
#print(inputY)        

C = torch.randn([27,2], device=DEVICE, requires_grad=True)
W1 = torch.randn([6,100], device=DEVICE, requires_grad=True)
B1 = torch.randn([100], device=DEVICE, requires_grad=True)
W2 = torch.randn([100,27], device=DEVICE, requires_grad=True)
B2 = torch.randn([27], device=DEVICE, requires_grad=True)
parameters = [C, W1, B1, W2, B2]

vel = [torch.zeros_like(param, device=DEVICE) for param in parameters]

for iteration in range(10000):
    emb = C[inputX]
    R = emb.view(-1, 6) @ W1 + B1
    R = R @ W2 + B2
    R = F.cross_entropy(R, inputY)

    if iteration % 1 == 0:
        for param in parameters:
            param.grad = None
        R.backward()
        with torch.no_grad():
            for param, v in zip(parameters, vel):
                v = 0.9 * v + param.grad
                param -= 0.1 * v
        if iteration % 100 == 0:
            print(iteration, R.item())

print("Final loss:", R.item())        

    


0 37.15438461303711
100 3.0024895668029785
200 2.8320391178131104
300 2.690033197402954
400 2.6465635299682617
500 2.6155612468719482
600 2.5967812538146973
700 2.580639123916626
800 2.5664830207824707
900 2.554378032684326
1000 2.544051170349121
1100 2.5361292362213135
1200 2.530522584915161
1300 2.5262649059295654
1400 2.520962715148926
1500 2.517160654067993
1600 2.5139808654785156
1700 2.51119065284729
1800 2.508690357208252
1900 2.5064401626586914
2000 2.5044057369232178
2100 2.5025548934936523
2200 2.5008602142333984
2300 2.499293804168701
2400 2.4978485107421875
2500 2.496495485305786
2600 2.495234251022339
2700 2.494058132171631
2800 2.492948532104492
2900 2.4919042587280273
3000 2.490920305252075
3100 2.4899814128875732
3200 2.4891035556793213
3300 2.488281011581421
3400 2.4874393939971924
3500 2.4863693714141846
3600 2.504758358001709
3700 2.4996860027313232
3800 2.495692014694214
3900 2.4932379722595215
4000 2.4914402961730957
4100 2.4900035858154297
4200 2.4888007640838623


KeyboardInterrupt: 

In [ ]:
def negLogLikelihood():
    emb = C[inputX]
    R = emb.view(-1, 6) @ W1 + B1
    R = R @ W2 + B2
    return F.cross_entropy(R, inputY)

def genWord():
    res = [0]*CONTEXT_WIDTH

    emb = C[res[-CONTEXT_WIDTH:]]
    R = emb.view(-1, 6) @ W1 + B1
    R = R @ W2 + B2
    R = F.softmax(R, dim=1)
    
    while True:
        emb = C[res[-CONTEXT_WIDTH:]]
        R = emb.view(-1, 6) @ W1 + B1
        R = R @ W2 + B2
        R = F.softmax(R, dim=1)
        b = R.multinomial(num_samples=1, replacement=True).item()
        res.append(b)
        if b == 0:
            break;

    return tensor_to_name(res[CONTEXT_WIDTH-1:])

print("loss:", negLogLikelihood().item())

for i in range(20):
    print(genWord())

# Leun    

negLogLikelihood: 2.4808077812194824
mati
chihinero
kialhinyelhogda
ryk
alenronihyaveri
ber
ymon
naclicdih
parsynastea
libtaphalojyax
har
resei
tacra
ririanrrinhenryhnab
dan
bleha
necii
kdanerodten
crrae

